> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 4 — Scientific Ingestion & Metadata (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Load heterogeneous sources (Markdown, CSV, JSON, text) with a uniform interface
- Build a corpus manifest with provenance for every record
- Attach typed metadata that survives chunking
- Normalize and de-duplicate before indexing

> Runtime: ~2 min (CPU)  
> Cost: $0 (optional LLM tagging gated)  
> Data: files from `data/`


Scientific RAG is only as good as its **ingestion layer**. Before embedding, load PDFs/Markdown/CSV/JSON into a single provenance-tracked corpus.

We build a production-shaped ingestion pipeline: load -> manifest -> typed metadata -> de-duplicate.

> Warning: ingestion bugs (silent parse failures, lost metadata) are the #1 cause of 'my RAG gives wrong answers.'


## API keys & credentials

Mostly local (no paid API needed); the bootstrap sets up an optional provider for gated LLM cells.


In [ ]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

## Installation (pinned)


In [ ]:
# @title Installing Python dependencies
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "pydantic>=2.6,<3" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [ ]:
# @title Setting LangSmith variables
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter4-ingestion"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")

## 1. A typed manifest record

Every ingested document becomes a `DocRecord` with provenance.


In [ ]:
# @title Importing libraries
from __future__ import annotations
import hashlib, json, datetime
from pathlib import Path
from typing import Any
from pydantic import BaseModel, Field

CORPUS_VERSION = "2026.07.21"


class DocRecord(BaseModel):
    doc_id: str
    source_path: str
    source_type: str
    text: str
    n_chars: int
    sha256: str
    ingested_at: str = Field(
        default_factory=lambda: datetime.datetime.now(datetime.timezone.utc).isoformat()
    )
    corpus_version: str = CORPUS_VERSION
    metadata: dict[str, Any] = Field(default_factory=dict)


def _sha(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]


def make_record(path, source_type, text, **extra):
    return DocRecord(
        doc_id=_sha(str(path) + text)[:12],
        source_path=str(path),
        source_type=source_type,
        text=text,
        n_chars=len(text),
        sha256=_sha(text),
        metadata={"filename": path.name, **extra},
    )


print("DocRecord ready (corpus_version", CORPUS_VERSION, ")")

## 2. Uniform loaders per source type


In [ ]:
def load_markdown(p):
    return make_record(p, "markdown", p.read_text(encoding="utf-8"))


def load_text(p):
    return make_record(p, "text", p.read_text(encoding="utf-8", errors="replace"))


def load_csv(p):
    import csv

    rows = list(csv.DictReader(p.open(encoding="utf-8")))
    text = "\n".join(" | ".join(f"{k}={v}" for k, v in r.items()) for r in rows)
    return make_record(p, "csv", text, n_rows=len(rows))


def load_json(p):
    obj = json.loads(p.read_text(encoding="utf-8"))
    return make_record(p, "json", json.dumps(obj, indent=2)[:5000])


LOADERS = {
    ".md": load_markdown,
    ".txt": load_text,
    ".csv": load_csv,
    ".json": load_json,
}
print("Loaders:", {k: v.__name__ for k, v in LOADERS.items()})

## 3. Run ingestion over a folder -> manifest


In [ ]:
DATA_DIR = Path("../../data/datasets")
records = []
for f in sorted(DATA_DIR.rglob("*")):
    if f.is_file() and f.suffix.lower() in LOADERS and f.stat().st_size < 2_000_000:
        try:
            records.append(LOADERS[f.suffix.lower()](f))
        except Exception as e:
            print("skipped", f.name, e)
print("Ingested", len(records), "documents")
for r in records[:6]:
    print(
        " ",
        r.doc_id,
        "|",
        r.source_type,
        "|",
        r.n_chars,
        "chars |",
        r.metadata.get("filename"),
    )

## 4. De-duplicate by content hash


In [ ]:
seen, unique = set(), []
for r in records:
    if r.sha256 not in seen:
        seen.add(r.sha256)
        unique.append(r)
print(len(records), "->", len(unique), "after de-dup")
records = unique

## 5. Persist the manifest (JSONL)


In [ ]:
manifest_path = Path("corpus_manifest.jsonl")
with manifest_path.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(r.model_dump_json() + "\n")
print("Wrote", manifest_path, "(", len(records), "records )")

## 6. (Optional) LLM auto-tagging - gated


In [ ]:
RUN_LLM_TAGGING = False
if RUN_LLM_TAGGING:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    for r in records[:3]:
        r.metadata["topic"] = llm.invoke(
            "One-word scientific topic for:\n" + r.text[:400]
        ).content
        print(r.doc_id, "->", r.metadata["topic"])
else:
    print("LLM tagging skipped (RUN_LLM_TAGGING=False).")

## Limitations & safety notes

- **Parse fidelity.** Real PDFs need `pypdf`/OCR and can silently drop tables/figures.
- **PHI/PII.** Clinical notes may contain protected health information - de-identify before indexing to external services.
- **Manifest != truth.** Provenance is recorded but scientific correctness is not validated.


---
### Further Reading

| Notebook | Relevance |
|----------|-----------|
| **Chapter 4 RAG** | Where ingested documents become retrievable |
| **Chapter 4 Chunking for Scientific Documents** | How chunking preserves metadata through ingestion |


In [ ]:
# Cleanup
import gc

for _v in ("records", "unique", "LOADERS"):
    globals().pop(_v, None)
try:
    Path("corpus_manifest.jsonl").unlink(missing_ok=True)
except Exception:
    pass
gc.collect()
print("Cleanup complete.")

## Exercises

<details><summary>Why store a SHA-256 content hash per document?</summary>Content-based de-duplication and change detection for re-indexing.</details>

<details><summary>Why keep metadata typed (Pydantic) instead of a loose dict?</summary>Validation fails fast at ingestion instead of corrupting the index.</details>

<details><summary>Risk of ingesting clinical notes without de-identification?</summary>Leaking PHI/PII into embeddings/vector stores and external APIs (HIPAA/GDPR).</details>

### Tasks
- **Task A** - Add a `load_pdf` loader using `pypdf`.
- **Task B** - Add an optional `doi` field populated from a filename pattern.
- **Task C** - Write `validate_manifest` asserting non-empty text and unique `doc_id`.
- **Task D** - Add a `section` extractor that splits Markdown on `#` headers.
